# TrainLM: first native TorchTPU training run

Open this notebook in Colab connected to a **single-device Cloud TPU v5e** runtime. It fine-tunes `distilgpt2` for three steps on a tiny in-memory text set to verify the public `TrainLMTrainer` path. It is a functional smoke, not a useful training recipe, performance result, or support certification.

This early route is opt-in with `accelerator="native_tpu"`. It requires exactly one visible TPU device. Multi-device training, compile, exact resume, and the declared Torch compatibility profile remain pending. It does not install packages or alter the runtime.


In [ ]:
from pathlib import Path
import subprocess
import sys
import tempfile

repo_root = Path(tempfile.mkdtemp(prefix="trainlm-native-tpu-")) / "TrainLM"
subprocess.run([
    "git", "clone", "--depth", "1", "--branch",
    "feat/01-native-torchtpu-single-device-training",
    "https://github.com/Dhiraj309/TrainLM.git", str(repo_root),
], check=True)
sys.path.insert(0, str(repo_root / "src"))
source_revision = subprocess.check_output(
    ["git", "-C", str(repo_root), "rev-parse", "HEAD"], text=True
).strip()
print(f"source_revision={source_revision}")


In [ ]:
# TorchTPU registers the native `tpu` device with PyTorch.
import torch_tpu  # noqa: F401
import torch
from importlib import metadata
from trainlm.runtime.torchtpu_discovery import discover_tpu

for package in ("torch", "torch-tpu", "libtpu", "transformers"):
    try:
        print(f"{package}={metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}=not-installed")

tpu = discover_tpu(torch, verify_allocation=True, include_properties=True)
print("native_tpu=", tpu)
if tpu["status"] != "ready" or tpu["device_count_visible"] != 1:
    raise RuntimeError("This notebook requires one allocatable native TPU device.")


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from trainlm import TrainLMTrainer, TrainLMTrainingArguments

model_id = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id)
model.config.pad_token_id = tokenizer.pad_token_id

texts = [
    "TrainLM trains causal language models with PyTorch.",
    "A TPU can accelerate training when the runtime is configured correctly.",
    "This small example checks the user-facing training path.",
    "Real experiments need a representative dataset and validated settings.",
] * 4
train_rows = []
for text in texts:
    encoded = tokenizer(text, max_length=64, truncation=True, padding="max_length", return_tensors="pt")
    input_ids = encoded["input_ids"][0]
    attention_mask = encoded["attention_mask"][0]
    labels = input_ids.clone()
    labels[attention_mask == 0] = -100
    train_rows.append({"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels})

args = TrainLMTrainingArguments(
    output_dir="/content/trainlm-native-tpu-smoke",
    accelerator="native_tpu",  # explicit experimental single-device route
    max_steps=3,
    per_device_train_batch_size=1,
    sequence_length=64,
    learning_rate=5e-5,
    logging_steps=1,
    bf16=False,  # establish eager fp32 behavior before testing reduced precision
)
trainer = TrainLMTrainer(
    model=model, tokenizer=tokenizer, train_dataset=train_rows, args=args
)
print(trainer.explain(format="text"))


In [ ]:
state = trainer.train()
print({"step": state.step, "loss": state.loss, "phase": state.phase.value})
export_path = trainer.save_model()
print(f"model_and_tokenizer_saved={export_path}")
print("result=single-device native trainer smoke; no compile, resume, distributed, or performance claim")


## If it fails

Keep the first exception and the printed package versions. Do not switch to CPU and label it a TPU pass. The current TrainLM project metadata excludes Torch 2.14.x, so a Torch 2.14 runtime may train through this experimental source path while remaining outside the declared compatibility contract. We will resolve that only after CPU conformance and clean-profile review.
